# Prompt anatomy: the same model, a very different result
**Role · task · context · constraints · output format · examples — and a prompt library you keep in Git**

Two jobs today. First, learn what a prompt is made of, and measure whether a change made it better. Second, create **your own work repository** and push your first prompts to it. Prompts are assets: they get versions, tests, and a history.

Before starting: `git pull`, `uv sync` (no new packages). Create your work repository first if you haven't (`module02/work_repo_steps.md`).

## 1. Setup

In [ ]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client                                   # Session 11
    from prompt_utils import (PromptSpec, check_work_repo, init_work_repo, results_table, save_to_library, show_tips)
    from prompt_tasks import (DEMO, TASK1, TASK2, TASK3, classify_spec, run_task, score_classifier)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11 and 14: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

client = groq_client()
MODEL = os.getenv("MODEL")
print("model:", MODEL)

## 2. Same model, two prompts
Nothing about the model changes in this section. Only the words we send. Each task has **acceptance checks**: small pieces of Python that say whether an output is good enough, like the acceptance criteria on a ticket. They turn "this looks better" into a number.

In [ ]:
def card(task):
    print(f"=== {task.id}: {task.title}\n{task.scenario}\n")
    print("INPUT you are given:\n" + task.user_input + "\n")
    if task.material:
        print("MATERIAL you may put in your prompt:\n" + "\n".join(task.material.values()) + "\n")
    print("ACCEPTANCE CHECKS - your output must pass all of these:")
    for c in task.acceptance():
        print("   -", c)
    print()

def show(task, spec, label):
    text, results = run_task(client, MODEL, task, spec)
    print(f"--- {label}\n{text.strip()}\n{results_table(results)}\n")
    return text, results

card(DEMO)
weak_out, weak_res = show(DEMO, DEMO.weak_spec(), 'WEAK prompt: "' + DEMO.weak_prompt + '"')
strong_out, strong_res = show(DEMO, DEMO.reference, "STRONG prompt (all five parts)")

*Question 1 — Which acceptance checks did the weak prompt fail? For each one, which missing part of the prompt would have prevented it?*

## 3. The parts, and where each one goes
A prompt has five parts, plus optional examples. `PromptSpec` holds them and shows how they become the **messages list** you already know from Sessions 2 and 11. Standing rules (role, constraints, format) go in the **system** message. The job itself (task, context, the actual input) goes in the **user** message.

In [ ]:
print("Parts missing from the weak prompt:")
show_tips(DEMO.weak_spec())
print("\nWhat the strong prompt actually sends:\n")
for m in DEMO.reference.to_messages(DEMO.user_input):
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

*Question 2 — Why do the rules live in the system message and the facts in the user message? What would change if you moved the policy text from context into the system message for every call?*

## 4. Zero-shot, one-shot, few-shot
Same instruction every time: classify a bank customer's message as card, loan, account or other. Only the number of **worked examples** changes (0, 1, 4). The examples teach the bank's *house convention*: UPI problems are "account", servicing requests are "other". Two scores per run: did it pick the right label, and did it obey "reply with the single word only"?

This makes 24 short calls. Results vary by model; read your own numbers.

In [ ]:
print(f"{'examples':>9} {'right label':>12} {'exact format':>13}")
for n in (0, 1, 4):
    r = score_classifier(client, MODEL, classify_spec(n))
    print(f"{n:>9} {r['accuracy']:>8}/{r['n']} {r['exact_format']:>9}/{r['n']}")
    for text, truth, got in r["wrong"]:
        print(f"            wrong: {text[:58]!r}  bank says {truth!r}, model said {got!r}")

*Question 3 — Did the examples change the label, the format, or both? A strong model may already score well with zero examples. When would you still pay for the extra tokens of few-shot on every call?*

## 5. Your turn: three real prompts
For each task: read the card, look at what the weak prompt produces, then fill in the parts in `my_spec_N` and rerun until every acceptance check passes. Change **one thing at a time** so you learn which change helped. Do not peek at `prompt_tasks.py`: the instructor's reference prompts are in there, and the learning is in trying first.

### Task 1 — a summary a credit committee can trust

In [ ]:
card(TASK1)
w1 = show(TASK1, TASK1.weak_spec(), 'WEAK prompt: "' + TASK1.weak_prompt + '"')

In [ ]:
my_spec_1 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK1.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (the lending policy is in TASK1.material["policy"])
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_1)
out1, res1 = show(TASK1, my_spec_1, "YOUR prompt, version 1")

### Task 2 — an SMS that must not leak

In [ ]:
card(TASK2)
w2 = show(TASK2, TASK2.weak_spec(), 'WEAK prompt: "' + TASK2.weak_prompt + '"')

In [ ]:
my_spec_2 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK2.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (and what must never be shared)
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_2)
out2, res2 = show(TASK2, my_spec_2, "YOUR prompt, version 1")

### Task 3 — JSON that a ticketing system can read

In [ ]:
card(TASK3)
w3 = show(TASK3, TASK3.weak_spec(), 'WEAK prompt: "' + TASK3.weak_prompt + '"')

In [ ]:
my_spec_3 = PromptSpec(
    role="",              # YOUR TURN: who is the model, and who is it writing for?
    task=TASK3.weak_prompt,   # sharpen this: one sentence, starts with a verb
    context="",           # facts and rules the model cannot know (the category and urgency definitions are in TASK3.material)
    constraints=[],       # limits: length, what must be included, what must never be revealed
    output_format="",     # the exact shape of the reply
)
show_tips(my_spec_3)
out3, res3 = show(TASK3, my_spec_3, "YOUR prompt, version 1")

*Question 4 — Which task needed the most iterations, and which single constraint or format line made the biggest difference? Which acceptance check would you add that we did not write?*

## 6. Your prompt library — a versioned asset
A prompt that works today will be edited next month by someone who forgot why it says what it says. So keep each version, the output it produced, and which checks it passed, in a file that lives in Git: `prompts/library.md` in **your work repository** (not the course repo). Never edit an old entry — add a new version. Git then gives you the history for free.

First, point the notebook at your cloned work repository.

In [ ]:
WORK_REPO = Path.home() / "Documents" / "genai-course-work"      # edit if you cloned it somewhere else
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if WORK_REPO.exists():
    print("created:", init_work_repo(WORK_REPO) or "nothing (all files were already there)")
else:
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

When all three of your prompts pass their checks, set `SAVE = True` and run the next cell **once**. (It stays off by default so that "Run All" never saves half-finished prompts.) If you improve a prompt later, bump `VERSION` and run again: a new entry is added, the old one stays.

In [ ]:
SAVE = False      # set to True when you are happy with all three prompts
VERSION = 1
if SAVE and WORK_REPO.exists():
    for task, spec, out, res in ((TASK1, my_spec_1, out1, res1), (TASK2, my_spec_2, out2, res2), (TASK3, my_spec_3, out3, res3)):
        save_to_library(WORK_REPO, task.id, VERSION, spec, out, res, purpose=task.title)
    print("saved 3 entries to", WORK_REPO / "prompts" / "library.md")
else:
    print("not saved (SAVE is False, or the work repo folder is missing)")

## 7. Commit and push — your first submission
In a terminal (not in this notebook), inside your **work** repository:

```powershell
cd ~\Documents\genai-course-work
git add .
git commit -m "Session 14: prompt library v1"
git push -u origin main        # the first time; afterwards just: git push
```
The first push opens a browser window to sign in to GitHub. Then run the self-check below. It reads your repository and tells you, in plain English, what is missing and the exact command to fix it.

In [ ]:
for r in check_work_repo(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

## 8. Reflection
Answer questions 1-4 above, one or two lines each, and write one line on what was hardest about the Git steps.

1.
2.
3.
4.

Checkpoint: every line of the self-check above says `PASS`, and `uv run pytest tests/test_s14.py` is green in the course repository. Then submit your repository link through the form.